# 05 Aggregate share classes and fund features

目标：以pass+unknown混合集合聚合TNA、净收益、费用、换手、flow和age，各变量使用自己的同一有效集合。

- 输入：04 shareclass eligibility
- 输出：fund_month_basic_features.parquet
- 主键：wficn+month
- 时间范围：规范锁定范围；smoke模式使用真实输入子样本
- 前置依赖：04
- 本 Notebook 不负责：下游阶段的排名、PCA或最终面板合并

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import polars as pl
from src.atomic_write import write_parquet_atomic
from src.runtime_paths import output_path, upstream_path
from src.schema_utils import assert_unique

frame=pl.read_parquet(upstream_path(ROOT,RUN_MODE,"04_filter_fund_universe","data/interim/04_fund_filter/shareclass_eligibility.parquet"))
frame=frame.filter(pl.col("eligible") & pl.col("eligibility_status").is_in(["pass","unknown"]))
def weighted(value, alias):
    valid=pl.col(value).is_not_null()&pl.col("lag_mtna").is_not_null()&(pl.col("lag_mtna")>0)
    numerator=pl.when(valid).then(pl.col(value)*pl.col("lag_mtna")).sum()
    denominator=pl.when(valid).then(pl.col("lag_mtna")).sum()
    return (numerator/denominator).alias(alias)
valid_flow=pl.col("mtna").is_not_null()&pl.col("lag_mtna").is_not_null()&pl.col("mret").is_not_null()&(pl.col("lag_mtna")>0)
valid_tna=pl.col("mtna").is_not_null()&(pl.col("mtna")>=0)
out_frame=frame.group_by(["wficn","month"],maintain_order=True).agg(
    weighted("mret","fund_return_net"),
    pl.when(valid_tna.any()).then(pl.when(valid_tna).then(pl.col("mtna")).sum()).otherwise(None).alias("tna"),
    (pl.when(valid_flow).then(pl.col("mtna")-pl.col("lag_mtna")*(1+pl.col("mret"))).sum()
     /pl.when(valid_flow).then(pl.col("lag_mtna")).sum()).alias("flow"),
    weighted("exp_ratio","exp_ratio"),
    weighted("turn_ratio","turnover"),
    pl.col("first_offer_dt").min().alias("_first_offer_dt"),
    pl.len().cast(pl.Int64).alias("n_aggregated_shareclasses"),
    (pl.col("eligibility_status")=="pass").sum().cast(pl.Int64).alias("n_pass_classes"),
    (pl.col("eligibility_status")=="unknown").sum().cast(pl.Int64).alias("n_unknown_classes"),
    pl.col("crsp_obj_cd").filter((pl.col("eligibility_status")=="pass")&pl.col("crsp_obj_cd").is_not_null()).unique().alias("_pass_styles"),
).with_columns(
    ((pl.col("month").dt.month_end()-pl.col("_first_offer_dt")).dt.total_days()/365.25).alias("age"),
    pl.when(pl.col("_pass_styles").list.len()==1).then(pl.col("_pass_styles").list.first()).otherwise(None).alias("fund_style"),
).with_columns(
    pl.when(pl.col("fund_return_net")>1).then(None).otherwise(pl.col("fund_return_net")).alias("fund_return_net")
).drop(["_first_offer_dt","_pass_styles"])
assert_unique(out_frame,["wficn","month"],"fund basic features")
out=output_path(ROOT,RUN_MODE,"05_aggregate_shareclasses_and_fund_features","data/processed/fund/fund_month_basic_features.parquet")
write_parquet_atomic(out_frame,out)
INPUTS_USED=[str(upstream_path(ROOT,RUN_MODE,"04_filter_fund_universe","data/interim/04_fund_filter/shareclass_eligibility.parquet").relative_to(ROOT))]
OUTPUTS_WRITTEN=[str(out.relative_to(ROOT))]; INPUT_COUNTS={"eligible_shareclasses":frame.height}; OUTPUT_COUNTS={"fund_months":out_frame.height}
PRIMARY_KEY=["wficn","month"]; DUPLICATE_KEY_COUNT=0; EXCLUDED_ROW_COUNT=0; EXCEPTION_FILE=None
STAGE_STATUS="passed"; STAGE_NOTES=["Each weighted variable used its own identical numerator/denominator valid set."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "05_aggregate_shareclasses_and_fund_features", stage="05", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
